<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Trees and ensembles

**[C05-M03-L01 · Trees and ensembles](https://learning.nextia-ai.com/courses/ml/m03/trees-and-ensembles/)** · C05, Machine Learning: From Problem to Reliable Model · Module 3, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** grow a decision tree by hand and with scikit-learn, combine many trees into a random forest and into gradient boosting, and compare them fairly with logistic regression by cross-validation.

| | |
|---|---|
| **Time** | About 65 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | The pipeline and `ticket_model.py`, from [Preprocess consistently](https://learning.nextia-ai.com/courses/ml/m02/preprocess-consistently/). Scores and `predict_proba`, from [Classification intuition](https://learning.nextia-ai.com/courses/ml/m02/classification-intuition/). |
| **You do not need** | The local project. The setup below downloads the data and writes `ticket_model.py`. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m03/trees-and-ensembles/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M03-L01-trees-and-ensembles/trees-and-ensembles-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the C05 practice data (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m01/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = []
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

## Setup: ticket_model.py

The next cell writes `ticket_model.py` as you wrote it in [Preprocess consistently](https://learning.nextia-ai.com/courses/ml/m02/preprocess-consistently/#write-ticket_modelpy): the column lists, `load()` and `build_pipeline()`. Run it. You should see `Writing ticket_model.py` (or `Overwriting`).

In [ ]:
%%writefile ticket_model.py
"""The shared parts of Larkfield's escalation model: the columns, how to load
a part of the data, and the preprocessing-plus-model pipeline.

Nextia Learning, C05, Module 2, lesson 3 (Preprocess consistently).
"""

from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA = Path("data")
CATEGORIES = ["channel", "team", "segment", "region"]
NUMBERS = ["priority", "order_value", "word_count", "customer_tenure_days", "prior_tickets_90d",
           "created_hour"]
FEATURES = CATEGORIES + NUMBERS
TARGET = "escalated_72h"


def read(path):
    """Read a ticket CSV file. Only an empty cell is missing: the text
    "unknown" stays a category."""
    return pd.read_csv(path, parse_dates=["created_at"], dtype={"customer_id": "string"},
                       keep_default_na=False, na_values=[""])


def load(part):
    """One part of the split: train, valid or test."""
    return read(DATA / f"{part}.csv")


def build_pipeline(model=None):
    """Preprocessing and model in one object. fit() learns the medians, the
    scaling and the categories from the rows it is given, and nothing else."""
    numbers = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ])
    prepare = ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore"), CATEGORIES),
        ("numbers", numbers, NUMBERS),
    ])
    return Pipeline([("prepare", prepare), ("model", model if model is not None else LogisticRegression(max_iter=1000))])

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Questions instead of weights

Logistic regression adds up the features, each with a weight. A **decision tree** works in another way: it asks questions about one feature at a time, such as "Is the priority 1?". Each answer sends a ticket left or right, to the next question. A **split** is one question and its two groups. A **leaf** is a group at the end, with no more questions. The tree's **score** for a ticket is the share of escalated train tickets in its leaf.

Run the next cell. It imports the tools and loads the train set.

In [ ]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text

from ticket_model import FEATURES, TARGET, build_pipeline, load

train = load("train")
print(len(train), "train tickets,", train[TARGET].sum(), "escalated")

> **Check.** You should see `16939 train tickets, 1983 escalated`.

Here are 10 made-up tickets, small enough to check by hand. Four of them escalated. Run the cell.

In [ ]:
tiny = pd.DataFrame({
    "ticket": [f"T-{i:02d}" for i in range(1, 11)],
    "priority": [1, 1, 1, 1, 2, 2, 2, 3, 3, 3],
    "word_count": [150, 40, 120, 45, 130, 160, 20, 30, 145, 60],
    "escalated": [1, 0, 1, 1, 1, 0, 0, 0, 0, 0],
})
print(tiny.to_string(index=False))

> **Check.** You should see 10 rows, T-01 to T-10. The escalated ones are T-01, T-03, T-04 and T-05.

## 2. Which question first?

A good question makes groups that are **pure**: all escalated, or all not escalated. **Impurity** measures how mixed a group is. The tree uses the Gini impurity: 2 × p × (1 − p), where p is the share of escalated tickets. A pure group has 0. A half-and-half group has 0.5, the most mixed.

Work it out by hand first, for the 10 tickets: p = 4/10 = 0.4, so the impurity is 2 × 0.4 × 0.6 = **0.48**.

> **Predict.** Two candidate questions are "priority = 1?" and "more than 100 words?". Which one makes purer groups? Write your guess, then run the cell.

In [ ]:
def impurity(escalated):
    """Gini impurity of a group: 2 × p × (1 − p), with p the share escalated."""
    p = escalated.mean()
    return 2 * p * (1 - p)


def impurity_after(question):
    """The impurity after a split: each group's impurity, weighted by its size."""
    yes, no = tiny[question], tiny[~question]
    return (len(yes) * impurity(yes["escalated"]) + len(no) * impurity(no["escalated"])) / len(tiny)


print("before any question: ", round(impurity(tiny["escalated"]), 3))
print("priority = 1?        ", round(impurity_after(tiny["priority"] == 1), 3))
print("more than 100 words? ", round(impurity_after(tiny["word_count"] > 100), 3))

> **Check.** You should see `0.48`, `0.317` and `0.4`.

"priority = 1?" makes 4 tickets with 3 escalated (impurity 0.375) and 6 tickets with 1 escalated (0.278). Weighted by size: (4 × 0.375 + 6 × 0.278) ÷ 10 = **0.317**. "More than 100 words?" makes two groups of 5, with 3 and 1 escalated: 0.40. The tree tries every possible question and takes the one with the lowest impurity after the split: here, "priority = 1?". Then it asks again inside each group, until it reaches its maximum depth. The **depth** is the number of questions from the top to a leaf.

Run the next cell. It grows a tree of depth 2 on the 10 tickets, prints its questions, and scores a new ticket with priority 2 and 140 words.

In [ ]:
tiny_tree = DecisionTreeClassifier(max_depth=2, random_state=0)
tiny_tree.fit(tiny[["priority", "word_count"]], tiny["escalated"])
print(export_text(tiny_tree, feature_names=["priority", "word_count"], show_weights=True))

new_ticket = pd.DataFrame({"priority": [2], "word_count": [140]})
print("score of the new ticket:", tiny_tree.predict_proba(new_ticket)[0, 1].round(3))

> **Check.** You should see `priority <= 1.50` first, then `word_count <= 42.50` and `word_count <= 95.00`. The score of the new ticket is `0.333`.

`weights: [2.00, 1.00]` means 2 tickets not escalated and 1 escalated in that leaf (T-05, T-06 and T-09). The new ticket lands there, so its score is 1 ÷ 3 = 0.333. The tree writes "priority 1" as `priority <= 1.50`: it puts the cut point halfway between two values.

## 3. The real tree

Now the same on the 16,939 train tickets, with the 10 features. A tree compares a feature with a cut point. Scaling changes the numbers but not their order, so the tickets go to the same side. The scaler does nothing for a tree, so `tree_pipeline()` takes it out (`"passthrough"` means "no step here").

Run the cell. The names have a prefix, such as `numbers__`, from the branch of the pipeline.

In [ ]:
def tree_pipeline(model):
    """build_pipeline() without the scaler: a tree does not need it."""
    return build_pipeline(model).set_params(prepare__numbers__scale="passthrough")


pipeline = tree_pipeline(DecisionTreeClassifier(max_depth=2, random_state=0))
pipeline.fit(train[FEATURES], train[TARGET])
names = list(pipeline.named_steps["prepare"].get_feature_names_out())
print(export_text(pipeline.named_steps["model"], feature_names=names, show_weights=True))

> **Check.** You should see `numbers__priority <= 1.50` first, then `numbers__word_count <= 66.50` and `numbers__word_count <= 100.50`. The four leaves have `[635.00, 128.00]`, `[1057.00, 565.00]`, `[7954.00, 507.00]` and `[5310.00, 783.00]`.

The tree found priority and the length of the first message. Priority 1 tickets with more than 66 words escalate most: 565 of 1,622, a score of 0.348. The [lesson page](https://learning.nextia-ai.com/courses/ml/m03/trees-and-ensembles/#the-real-tree) draws this tree.

## 4. How to compare models fairly

**Average precision (AP)** is the course's main score for a ranking (Module 4 explains it fully). Sort the tickets by score, highest first. Each time you reach an escalated ticket, compute the precision so far. AP is the mean of these precisions. 1.0 means all escalated tickets come first. One score for every ticket gives the escalation rate.

Five tickets, sorted: escalated, not, escalated, not, not. The precisions at the escalated ones are 1/1 and 2/3, so AP = (1 + 0.667) ÷ 2 = 0.833. Run the cell.

In [ ]:
truth = [1, 0, 1, 0, 0]
scores = [0.9, 0.8, 0.7, 0.3, 0.1]
print("AP of the five tickets:", round(average_precision_score(truth, scores), 3))

same_score = [0.117] * len(train)
print("AP of one score for every train ticket:", round(average_precision_score(train[TARGET], same_score), 3))

> **Check.** You should see `0.833` and `0.117`, the escalation rate of train.

You choose between models with the train set only. You keep the validation month to confirm the choice. But a model must not be scored on the tickets it learned from. **Cross-validation** (CV) solves this. It cuts train into 5 parts, the **folds**. It fits the model 5 times. Each time, one fold is the test, and the model learns from the other four. *Stratified* means that every fold has the same escalation rate. Run the cell.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
for fold, (learn_rows, test_rows) in enumerate(cv.split(train, train[TARGET]), start=1):
    rate = train[TARGET].iloc[test_rows].mean()
    print(f"fold {fold}: learn from {len(learn_rows)} tickets, test on {len(test_rows)}, escalated {rate:.3f}")

> **Check.** You should see 5 folds. Each learns from 13,551 or 13,552 tickets and tests on 3,387 or 3,388. The rate is 0.117 in every fold.

Every train ticket is tested exactly once, by a model that did not learn from it. Now score logistic regression, the model of Module 2, on the 5 folds.

In [ ]:
fold_scores = cross_val_score(build_pipeline(), train[FEATURES], train[TARGET],
                              cv=cv, scoring="average_precision")
print("AP on each fold:", fold_scores.round(3))
print(f"CV AP {fold_scores.mean():.3f} ± {fold_scores.std():.3f}")

> **Check.** You should see `[0.316 0.31  0.342 0.339 0.327]` and `CV AP 0.327 ± 0.012`.

The 5 scores differ by up to 0.032, only because the folds have different tickets. The **standard deviation** (std) of the 5 scores, 0.012, measures that spread. The rule of this module: **a difference smaller than the spread is not a real difference.** [Validation strategy](https://learning.nextia-ai.com/courses/ml/m04/validation-strategy/) in Module 4 explains cross-validation fully.

## 5. Worked example: logistic regression against three trees

`evaluate()` returns two numbers for a pipeline: the train AP (fit on all train tickets, scored on the same tickets) and the CV AP.

> **Predict.** Which tree has the highest train AP? Which has the highest CV AP? Then run the cell. It takes a few seconds.

In [ ]:
def evaluate(pipeline):
    """Train AP (fit on all train tickets) and CV AP (mean ± std of the 5 folds)."""
    fold_scores = cross_val_score(pipeline, train[FEATURES], train[TARGET], cv=cv, scoring="average_precision")
    pipeline.fit(train[FEATURES], train[TARGET])
    train_ap = average_precision_score(train[TARGET], pipeline.predict_proba(train[FEATURES])[:, 1])
    return {"train AP": round(float(train_ap), 3), "CV AP": round(float(fold_scores.mean()), 3),
            "std": round(float(fold_scores.std()), 3)}


def table(results):
    """The results as a table, one row per model (rows still set to ... are left out)."""
    return pd.DataFrame({name: r for name, r in results.items() if isinstance(r, dict)}).T


results = {
    "logistic regression": evaluate(build_pipeline()),
    "tree, depth 3": evaluate(tree_pipeline(DecisionTreeClassifier(max_depth=3, random_state=0))),
    "tree, depth 6": evaluate(tree_pipeline(DecisionTreeClassifier(max_depth=6, random_state=0))),
    "tree, no limit": evaluate(tree_pipeline(DecisionTreeClassifier(random_state=0))),
}
print(table(results))

> **Check.** You should see these rows:
>
> | | train AP | CV AP | std |
> |---|---|---|---|
> | logistic regression | 0.328 | 0.327 | 0.012 |
> | tree, depth 3 | 0.244 | 0.234 | 0.011 |
> | tree, depth 6 | 0.346 | 0.258 | 0.010 |
> | tree, no limit | 1.000 | 0.137 | 0.004 |

The tree with no limit is perfect on train (1.000) and the worst on tickets it did not see (0.137): it memorised the train tickets. Lesson 2 is about this. No tree beats logistic regression.

## 6. Many trees: a random forest

One deep tree changes a lot when its train tickets change. A **random forest** grows many trees and averages their scores. Two things make the trees different:

- Each tree learns from a **bootstrap sample**: as many tickets as train has, drawn at random *with replacement*. Some tickets come twice, some not at all.
- At each split, a tree looks at a random part of the features only.

An **ensemble** is a model made of many models. Run the cell: a forest of 5 small trees on the 10 tickets.

In [ ]:
tiny_forest = RandomForestClassifier(n_estimators=5, max_depth=2, random_state=0)
tiny_forest.fit(tiny[["priority", "word_count"]], tiny["escalated"])
print("tree 1 learned from:", sorted(tiny["ticket"].iloc[tiny_forest.estimators_samples_[0]]))

new_ticket = pd.DataFrame({"priority": [1], "word_count": [100]})
each_tree = [round(float(tree.predict_proba(new_ticket.to_numpy())[0, 1]), 2) for tree in tiny_forest.estimators_]
print("score of each tree:", each_tree)
print("forest score (the average):", round(float(tiny_forest.predict_proba(new_ticket)[0, 1]), 2))

> **Check.** Tree 1 learned from T-01, T-03, T-04 and T-10 twice each, and from T-07 and T-08 once. T-02, T-05, T-06 and T-09 are not in its sample. The scores of the trees are `[1.0, 0.4, 1.0, 1.0, 1.0]`, and the forest score is `0.88`, their average.

One tree disagrees. The average is steadier than any single tree.

## 7. Trees that fix errors: gradient boosting

**Gradient boosting** also builds many small trees, but one after another. Each new tree learns the errors that the trees before it still make, and adds a part of its correction. The **learning rate** sets that part.

The next cell does two steps by hand. Every ticket starts at 0.4, the escalation rate. Each small tree has one question and learns the error, *escalated − score*. (Real boosting works on another scale, so a score stays between 0 and 1. The idea is the same.) Run the cell.

In [ ]:
x_tiny = tiny[["priority", "word_count"]]
boost = tiny[["ticket", "escalated"]].copy()
boost["start"] = 0.4                                # the escalation rate, for every ticket
score = boost["start"]
for step in (1, 2):
    error = tiny["escalated"] - score               # what the scores still miss
    small_tree = DecisionTreeRegressor(max_depth=1).fit(x_tiny, error)
    print(f"tree {step} asks:", export_text(small_tree, feature_names=["priority", "word_count"]).split()[1:4])
    score = score + 0.5 * small_tree.predict(x_tiny)  # learning rate 0.5: half of the correction
    boost[f"after tree {step}"] = score.round(3)
print(boost.to_string(index=False))

> **Check.** Tree 1 asks about `priority <= 1.50`. After tree 1, the priority 1 tickets have 0.575 and the others 0.283. Tree 2 asks about `word_count <= 42.50`. After tree 2, T-01, T-03 and T-04 have 0.657, and T-07 and T-08 have 0.093.

Tree 2 corrects tree 1: T-02 is priority 1 but short and not escalated, so its score goes down (0.385). `HistGradientBoostingClassifier`, the scikit-learn version for tables, adds up to 100 trees in this way.

## 8. Guided practice: add the forest and boosting

> **Predict.** Will a forest of 200 trees beat logistic regression's CV AP of 0.327?

> **Your turn.** Replace `...` with `evaluate(tree_pipeline(RandomForestClassifier(...)))`: 200 trees, `min_samples_leaf=20`, `n_jobs=-1`, `random_state=0`. `min_samples_leaf=20` stops a split that leaves fewer than 20 tickets in a leaf. `n_jobs=-1` uses all processor cores. Run the cell, then the check cell. It takes about 10 seconds.

In [ ]:
# Your turn: a random forest with 200 trees, min_samples_leaf=20, n_jobs=-1 and random_state=0.
results["random forest, min leaf 20"] = ...
print(table(results))

In [ ]:
entry = results.get('random forest, min leaf 20')
if not isinstance(entry, dict):
    print("Not yet: replace ... with evaluate(tree_pipeline(...)), then run the cell again.")
else:
    expect("the forest's CV AP", entry["CV AP"], 0.325)

> **Check.** The new row is `0.483  0.325  0.018`.

> **Your turn.** Now add gradient boosting with the default settings: `HistGradientBoostingClassifier(random_state=0)`. Run the cell, then the check cell.

In [ ]:
# Your turn: gradient boosting with the default settings and random_state=0.
results["boosting, default"] = ...
print(table(results))

In [ ]:
entry = results.get('boosting, default')
if not isinstance(entry, dict):
    print("Not yet: replace ... with evaluate(tree_pipeline(...)), then run the cell again.")
else:
    expect('the boosting CV AP', entry["CV AP"], 0.316)

> **Check.** The new row is `0.477  0.316  0.010`.

Run the next cell. It draws each CV AP with a bar of one std on each side, and a dashed line for one score for all tickets (0.117).

In [ ]:
import matplotlib.pyplot as plt

cv_table = table(results)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.errorbar(cv_table["CV AP"], cv_table.index, xerr=cv_table["std"], fmt="o", capsize=4)
ax.axvline(train[TARGET].mean(), linestyle="--", color="grey")
ax.text(train[TARGET].mean() + 0.003, -0.4, "one score for all", color="grey")
ax.set_xlabel("CV average precision (mean ± one std)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

> **Check.** You should see six points. The bars of logistic regression, the forest and boosting overlap. The tree with no limit is far to the left.

## 9. What the table says

The ensembles fix most of what a single tree does wrong: 0.325 and 0.316, against 0.258 for the best single tree. But they only **tie** logistic regression (0.327 ± 0.012). The differences are smaller than the spread. The real pattern here is mostly **additive**: each fact adds or removes some risk. Logistic regression has exactly that shape, so the trees have nothing extra to find.

Priya now checks the validation month once, as a confirmation, not to choose. Run the cell.

In [ ]:
valid = load("valid")
check_month = {
    "logistic regression": build_pipeline(),
    "random forest, min leaf 20": tree_pipeline(
        RandomForestClassifier(n_estimators=200, min_samples_leaf=20, n_jobs=-1, random_state=0)),
    "boosting, default": tree_pipeline(HistGradientBoostingClassifier(random_state=0)),
}
for name, pipeline in check_month.items():
    pipeline.fit(train[FEATURES], train[TARGET])
    scores = pipeline.predict_proba(valid[FEATURES])[:, 1]
    print(f"{name:28s} valid AP {average_precision_score(valid[TARGET], scores):.3f}")

> **Check.** You should see `0.353` for logistic regression, `0.382` for the forest and `0.369` for boosting.

The forest is 0.029 higher on May. May has only 183 escalations, so its AP moves by about 0.03 by chance alone. The difference is inside that noise, and CV with 16,939 tickets said "tie". A tie is a reason to keep the simpler model.

## 10. Your turn: a slower, smaller boosting

Change one condition: give boosting smaller trees (`max_depth=3`) and smaller steps (`learning_rate=0.05`).

> **Predict.** Will the train AP go up or down from 0.477? Will the CV AP change much?

> **Your turn.** Replace `...`, run the cell, then the check cell.

In [ ]:
# Your turn: boosting with max_depth=3 and learning_rate=0.05 (and random_state=0).
boosting_slow = ...
print(boosting_slow)

In [ ]:
if not isinstance(boosting_slow, dict):
    print("Not yet: replace ... with evaluate(tree_pipeline(HistGradientBoostingClassifier(...))).")
else:
    expect_hash('your result', boosting_slow, '47f7a8a81fb404eb')

Compare your numbers with the answer on the [lesson page](https://learning.nextia-ai.com/courses/ml/m03/trees-and-ensembles/#your-turn-a-slower-smaller-boosting). The train AP falls a lot, and the CV AP moves less than one std.

## 11. Failure case: the perfect score

Tomás tries a random forest with the default settings. Run the cell. It takes about 10 seconds.

In [ ]:
forest_default = evaluate(tree_pipeline(RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0)))
print("random forest, default settings:", forest_default)

> **Check.** You should see `train AP 1.0`, `CV AP 0.29` and `std 0.014`.

A train AP of 1.000 looks perfect. But the default trees grow until every leaf is pure, so each tree memorises its bootstrap sample, and the forest scores its own train tickets almost exactly. On tickets it did not see, it is lower than the forest with `min_samples_leaf=20` (0.325). **Never judge a model by its score on the tickets it learned from.** The [lesson page](https://learning.nextia-ai.com/courses/ml/m03/trees-and-ensembles/#the-perfect-score) has the full failure case.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/ml/m03/trees-and-ensembles/#knowledge-check). They count toward your certificate when you are signed in and enrolled.

In this lesson you grew a tree by impurity, combined trees by averaging (a forest) and by correcting errors (boosting), and compared every model with the same 5-fold CV. The ensembles tie logistic regression on this data. A deep tree memorises the train set: the next lesson measures that and controls it.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Control complexity](https://learning.nextia-ai.com/courses/ml/m03/control-complexity/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m03/trees-and-ensembles/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.